# Model training walkthrough

Step-by-step version of `python -m src.train`, using the same functions as the production pipeline.

> **Educational demo only.** Predictions are statistical estimates, **not medical diagnoses**. If `data/disease_symptoms.csv` is the generated demo file, the data is **synthetic** and the scores below say nothing about real-world accuracy.

In [ ]:
import sys, json, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():          # notebook started from notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.metrics import classification_report

from src import config
from src.preprocessing import load_dataset, clean_dataset, split_data
from src.train import build_candidates, make_pipeline, compute_metrics, permutation_importances

pd.set_option("display.max_columns", 50)

## 1. Load the dataset

In [ ]:
raw = load_dataset(config.DATA_PATH)
print(raw.shape)
raw.head()

## 2. Clean and preprocess

Duplicates are removed *before* splitting so identical records cannot land in both train and test. Missing values are reported here and imputed later **inside the pipeline** (fitted on training rows only).

In [ ]:
df, schema, report = clean_dataset(raw, config.MIN_SAMPLES_PER_CLASS)
print(json.dumps({k: v for k, v in report.items() if k != "class_counts"}, indent=2))
print("Uses age:", schema.has_age, "| uses gender:", schema.has_gender)

## 3. Class balance

In [ ]:
counts = df["disease"].value_counts()
ax = counts.sort_values().plot.barh(figsize=(7, 4), color="#4636C9")
ax.set_xlabel("Records"); ax.set_title(f"Class distribution (largest:smallest = {counts.max()/counts.min():.1f}:1)")
plt.tight_layout(); plt.show()

## 4. Reproducible stratified split

In [ ]:
split = split_data(df, schema, config.TEST_SIZE, config.VAL_SIZE, config.RANDOM_STATE)
print({name: len(getattr(split, name)) for name in ["X_train", "X_val", "X_test"]})

## 5. Train and compare models on the validation set

In [ ]:
labels = sorted(df["disease"].unique())
fitted, rows = {}, []
for name, estimator in build_candidates(config.RANDOM_STATE).items():
    pipe = make_pipeline(schema, estimator).fit(split.X_train, split.y_train)
    m = compute_metrics(split.y_val, pipe.predict(split.X_val), labels)
    fitted[name] = pipe
    rows.append({"model": name, **{k: m[k] for k in ["accuracy", "precision_macro", "recall_macro", "f1_macro"]}})
comparison = pd.DataFrame(rows).set_index("model")
comparison.round(3)

## 6. Select by validation macro-F1, refit on train + validation, evaluate once on the test set

In [ ]:
best = comparison.sort_values(["f1_macro", "accuracy"]).index[-1]
print("Selected:", best)
final = clone(fitted[best]).fit(split.X_dev, split.y_dev)
test = compute_metrics(split.y_test, final.predict(split.X_test), labels)
print({k: round(test[k], 3) for k in ["accuracy", "precision_macro", "recall_macro", "f1_macro"]})
print(classification_report(split.y_test, final.predict(split.X_test), zero_division=0))

In [ ]:
cm = np.array(test["confusion_matrix"])
plt.figure(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion matrix (test set)")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

## 7. Which inputs matter?

Permutation importance is model-agnostic. It describes what *this model* relies on - not medical causes.

In [ ]:
imp = pd.DataFrame(permutation_importances(final, split.X_test, split.y_test, config.RANDOM_STATE)).head(15)
imp.iloc[::-1].plot.barh(x="feature", y="importance", legend=False, figsize=(7, 5), color="#4636C9")
plt.xlabel("Drop in macro-F1 when shuffled"); plt.tight_layout(); plt.show()

## 8. Persist the model

The notebook does not overwrite your deployed model. To save a versioned model that the web app will load, run in a terminal:

```bash
python -m src.train
```

## Reading the results responsibly

* A single hold-out split has sampling noise; small differences between models are not meaningful.
* High accuracy on a dataset does **not** make a system medically reliable - real patients, rarer diseases and unreported symptoms are not represented.
* On synthetic or templated data, near-perfect scores are expected and uninformative.